<a href="https://colab.research.google.com/github/nattaponm/Teach_Nowcast_Pysteps_TMD/blob/main/notebooks/04_probabilistic_nowcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4: Probabilistic nowcasting และความน่าจะเป็นของฝนหนักที่เชียงของ
### STEPS ensemble, exceedance probability, point warnings and ensemble verification

⏱️ เวลาเรียนโดยประมาณ **90 นาที** | ต้องผ่าน: Notebook 0–3

---
## 🎯 วัตถุประสงค์การเรียนรู้
เมื่อจบ notebook นี้ นิสิตจะสามารถ
1. **สร้าง ensemble nowcast** ด้วย STEPS บนข้อมูล TMD (time step 15 นาที)
2. **ทำแผนที่ความน่าจะเป็น** $P(R > r)$ และ **กราฟความน่าจะเป็น ณ จุด** (อ.เชียงของ) สำหรับการเตือนภัย
3. **อธิบายแนวคิด neighbourhood probability** ซึ่งเหมาะกับพายุเซลล์เล็ก
4. **ประเมิน ensemble** ด้วย CRPS, ROC, reliability diagram และ rank histogram จากหลาย issue time

## 📖 1. แนวคิดและสมการ

### 1.1 STEPS (ทบทวน)
$$
\Psi_j^{(m)}(t) = \phi_{j,1}\Psi_j^{(m)}(t-1) + \phi_{j,2}\Psi_j^{(m)}(t-2) + \sigma_j\,\varepsilon_j^{(m)}(t)
$$
noise $\varepsilon_j^{(m)}$ ทำให้แต่ละ member $m$ มีรายละเอียดเหมือนฝนจริง และ ensemble สะท้อนความไม่แน่นอนของการเกิด/สลายของเซลล์ ซึ่งเป็นแหล่งความผิดพลาดหลักที่พบใน NB2–3

### 1.2 ความน่าจะเป็นแบบ pixel และแบบ neighbourhood
**Pixel probability:**
$$
P\big(R(\mathbf{x}) > r\big) \approx \frac{1}{M}\sum_{m=1}^{M} \mathbf{1}\left[R^{(m)}(\mathbf{x}) > r\right]
$$

**Neighbourhood probability:** โอกาสที่ฝนจะเกิน $r$ **ที่ใดที่หนึ่งภายในรัศมี $d$** รอบจุดสนใจ ซึ่งเหมาะกับคำเตือนระดับพื้นที่ (เช่น อำเภอ) มากกว่า เพราะตำแหน่งของเซลล์เล็กพยากรณ์ได้ไม่แม่นยำ

$$
P_{d}(\mathbf{x}_0) \approx \frac{1}{M}\sum_{m=1}^{M} \mathbf{1}\left[\max_{|\mathbf{x}-\mathbf{x}_0| \le d} R^{(m)}(\mathbf{x}) > r\right]
$$

### 1.3 Probabilistic verification (ทบทวน)
| เครื่องมือ | ตอบคำถาม |
|---|---|
| **CRPS** | ความผิดพลาดของการกระจายทั้งหมด (deterministic = MAE) |
| **ROC** | แยกเหตุการณ์ได้ดีแค่ไหน (discrimination) |
| **Reliability** | ความน่าจะเป็นตรงกับความถี่ที่เกิดจริงหรือไม่ |
| **Rank histogram** | spread ของ ensemble เหมาะสมหรือไม่ |

## ⚙️ 2. เตรียมสภาพแวดล้อมและข้อมูล

In [ ]:
#@title ⚙️ Setup: ติดตั้ง pysteps และดาวน์โหลดข้อมูล TMD (รันก่อนเสมอ ~2-3 นาทีใน Colab)
import os, sys, warnings
warnings.filterwarnings("ignore")

REPO = "Teach_Nowcast_Pysteps_TMD"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pysteps==1.21.5 cartopy netCDF4 rasterio geopandas
    if not os.path.exists(REPO):
        !git clone -q --depth 1 https://github.com/nattaponm/{REPO}.git

# หาโฟลเดอร์หลักของ repo (ที่มี utils/ และ data/)
for ROOT in [REPO, ".", ".."]:
    if os.path.exists(os.path.join(ROOT, "utils", "tmd_tools.py")):
        break
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")

import numpy as np
import matplotlib.pyplot as plt
import pysteps
from utils.data_io import load_radar_nc
from utils import mapplot as mp
from utils import tmd_tools as tt
mp.set_style()
mp.BASEMAP.update(provinces=True, rivers=True, lakes=False)   # จังหวัด + แม่น้ำโขง
mp.POINTS[:] = tt.POIS                                         # เรดาร์ เชียงของ เชียงแสน
print("✅ พร้อมใช้งาน | ข้อมูลอยู่ที่:", DATA)

In [ ]:
import contextlib, io, time
from pysteps import motion, nowcasts, verification
from pysteps.postprocessing.ensemblestats import excprob
from pysteps.utils import transformation

R, meta = load_radar_nc(os.path.join(DATA, "tmd_cri_20200423_rain_1km.nc"))
ts = meta["timestamps"]
DT = meta["accutime"]
BOX = tt.range_box(meta, 165)
CK_BOX = tt.zoom_box(meta, *tt.CHIANG_KHONG, half_width_km=70)
radar_mask = np.isfinite(R[0])

def to_dbr(x):
    d, m = transformation.dB_transform(x, meta, threshold=0.1, zerovalue=-15.0)
    d[~np.isfinite(d)] = -15.0
    return d, m

def to_mmh(d, m):
    r = transformation.dB_transform(d, threshold=m["threshold"], inverse=True)[0]
    r[..., ~radar_mask] = np.nan
    return r

steps = nowcasts.get_method("steps")
def run_steps(i, n_lead, n_ens=20, seed=24, quiet=True):
    """STEPS ensemble ออก ณ ts[i] (ใช้ภาพ i-3 … i) คืนค่า (member, lead, y, x) mm/h"""
    d, m = to_dbr(R[i - 3:i + 1])
    V = motion.get_method("LK")(d, verbose=False)
    sink = contextlib.redirect_stdout(io.StringIO()) if quiet else contextlib.nullcontext()
    with sink:
        e = steps(d[-3:], V, n_lead, n_ens_members=n_ens, n_cascade_levels=6,
                  precip_thr=m["threshold"], kmperpixel=meta["xpixelsize"] / 1000,
                  timestep=DT, noise_method="nonparametric", vel_pert_method="bps",
                  mask_method="incremental", seed=seed, num_workers=2)
    return to_mmh(e, m), V

## 💻 3. Ensemble nowcast ออก ณ 11:45 UTC (18:45 น.)
ณ เวลานี้ พายุลูกเห็บกำลังอยู่เหนือเชียงของ (ดูกราฟใน NB1 ข้อ 5) และจะเคลื่อนออกไปทางตะวันออกหลัง 12:15 UTC เราจะพยากรณ์ล่วงหน้า 5 time steps (+15 ถึง +75 นาที, 12:00–13:00 UTC) เพื่อดูว่า nowcast บอกได้หรือไม่ว่า "พายุจะผ่านไปเมื่อไร"

In [ ]:
I0, N_LEAD = 3, 5
tic = time.time()
ens, V = run_steps(I0, N_LEAD, n_ens=20, quiet=False)
obs = R[I0 + 1:I0 + 1 + N_LEAD]
lead_min = (np.arange(N_LEAD) + 1) * DT
ens_mean = np.nanmean(ens, axis=0)
print(f"\nensemble: {ens.shape} (member, lead, y, x) | ใช้เวลา {time.time()-tic:.0f} s")

In [ ]:
cols = [("Observed", lambda k: obs[k]), ("Ensemble mean", lambda k: ens_mean[k]),
        ("Member 1", lambda k: ens[0, k]), ("Member 2", lambda k: ens[1, k])]
leads = [1, 3]                                    # +30, +60
fig, axs = mp.map_panels(2, 4, meta, panel_width=2.5, zoom=CK_BOX)
for r, k in enumerate(leads):
    for c, (name, get) in enumerate(cols):
        lab = tuple(s for s, ok in (("left", c == 0), ("bottom", r == 1)) if ok)
        im = mp.radar_map(axs[r * 4 + c], get(k), meta, title=f"{name} +{int(lead_min[k])} min",
                          zoom=CK_BOX, gridline_labels=lab, north=(r == 0 and c == 0),
                          scalebar=(c == 0), points=(c == 0))
mp.add_colorbar(fig, im, axs)
mp.panel_labels(axs)
mp.save_figure(fig, "nb4_fig01_steps_members")
plt.show()

## 🎲 4. แผนที่ความน่าจะเป็น $P(R > 5\ \mathrm{mm\,h^{-1}})$
(5 mm/h ≈ 33 dBZ ด้วย Marshall–Palmer) เส้นสีดำคือขอบเขตฝน > 5 mm/h ที่เกิดขึ้นจริง

In [ ]:
THR = 5.0
fig, axs = mp.map_panels(1, 3, meta, panel_width=3.0, zoom=CK_BOX)
for j, k in enumerate([0, 1, 3]):
    P = excprob(ens[:, k], THR, ignore_nan=True)
    P[~radar_mask] = np.nan
    im = mp.radar_map(axs[j], P, meta, ptype="prob", probthr=THR, zoom=CK_BOX,
                      title=f"+{int(lead_min[k])} min ({ts[I0+1+k]:%H:%M} UTC)",
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",), north=(j == 0))
    mp.obs_contour(axs[j], obs[k], meta, level=THR, label=f"Observed > {THR:g} mm/h" if j == 2 else None)
mp.add_colorbar(fig, im, axs)
mp.panel_labels(axs)
mp.save_figure(fig, "nb4_fig02_exceedance_probability")
plt.show()

## 📍 5. ผลิตภัณฑ์เตือนภัย: ความน่าจะเป็น ณ อ.เชียงของ
เปรียบเทียบความน่าจะเป็นแบบ **pixel** (ที่ตั้งอำเภอ) กับแบบ **neighbourhood** (รัศมี 5 และ 10 km) และค่าสังเกตจริง (ฝนสูงสุดในรัศมี 5 km)

In [ ]:
import matplotlib.dates as mdates
x0, y0 = tt.lonlat_to_xy(meta, *tt.CHIANG_KHONG)
c0, r0 = (int(round(v)) for v in tt.xy_to_pixel(meta, x0, y0))
p_pixel = [np.mean(ens[:, k, r0, c0] > THR) for k in range(N_LEAD)]
p_nb = {}
for d_km in (5, 10):
    mx = tt.point_series(ens, meta, *tt.CHIANG_KHONG, radius_km=d_km, stat="max")   # (member, lead)
    p_nb[d_km] = np.mean(mx > THR, axis=0)
obs_ck = tt.point_series(R, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max")
valid = ts[I0 + 1:I0 + 1 + N_LEAD]

fig, ax1 = plt.subplots(figsize=(6, 3.2), layout="constrained")
ax1.plot(valid, p_pixel, "o-", color="#fdae61", label="P, pixel")
ax1.plot(valid, p_nb[5], "s-", color="#f46d43", label="P, 5-km neighbourhood")
ax1.plot(valid, p_nb[10], "^-", color="#a50026", label="P, 10-km neighbourhood")
ax1.set_ylim(0, 1.05)
ax2 = ax1.twinx()
ax2.bar(ts, obs_ck, width=0.006, color="0.75", alpha=0.7, label="Observed max (5 km)", zorder=0)
ax2.axhline(THR, color="k", ls=":", lw=0.8)
ax2.set_ylabel("Observed max rain rate (mm h$^{-1}$)")
ax1.set_zorder(ax2.get_zorder() + 1); ax1.patch.set_visible(False)
ax1.axvline(ts[I0], color="k", lw=1)
ax1.text(ts[I0], 0.02, " issue time", va="bottom", ha="left", fontsize=7)
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
mp.skill_axes(ax1, xlabel="Time (UTC)", ylabel=f"P(R > {THR:g} mm h$^{{-1}}$)",
              title="Probabilistic nowcast for Chiang Khong")
ax1.legend(frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(0, 0.9))
mp.save_figure(fig, "nb4_fig03_chiang_khong_probability")
plt.show()

> 🔍 **สังเกต:**
> - ความน่าจะเป็นแบบ neighbourhood **สูงกว่าแบบ pixel เสมอ** (มีโอกาสเกิด "ที่ใดที่หนึ่งในรัศมี" มากกว่า "ที่จุดนี้พอดี") จึงเหมาะกับคำเตือนระดับอำเภอ
> - ให้เปรียบเทียบกับแท่งสีเทา (ค่าสังเกต) ว่า nowcast บอกได้ถูกต้องหรือไม่ว่าพายุจะ "ผ่านพ้น" เชียงของเมื่อใด

## 📏 6. Verification ของ ensemble จาก 6 issue time (11:45–13:00 UTC)
⏱️ ใช้เวลาประมาณ 1.5–3 นาที (STEPS 20 members × 6 ครั้ง)

In [ ]:
ISSUES, NL = list(range(3, 9)), 4
crps = {"STEPS": np.zeros(NL), "Extrapolation": np.zeros(NL)}
roc = verification.ROC_curve_init(THR, n_prob_thrs=10)
rel = verification.reldiag_init(THR, n_bins=10)
rnk = verification.rankhist_init(20, THR)
extrap = nowcasts.get_method("extrapolation")
tic = time.time()
for i in ISSUES:
    e, Vi = run_steps(i, NL)
    d, m = to_dbr(R[i - 3:i + 1])
    det = to_mmh(extrap(d[-1], Vi, NL), m)
    for k in range(NL):
        o = R[i + 1 + k]
        crps["STEPS"][k] += verification.probscores.CRPS(e[:, k], o) / len(ISSUES)
        crps["Extrapolation"][k] += verification.probscores.CRPS(det[None, k], o) / len(ISSUES)
        P = excprob(e[:, k], THR, ignore_nan=True)
        verification.ROC_curve_accum(roc, P, o)
        verification.reldiag_accum(rel, P, o)
        verification.rankhist_accum(rnk, e[:, k], o)
print(f"เสร็จใน {time.time()-tic:.0f} s")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(14, 3.3), layout="constrained")
lm = (np.arange(NL) + 1) * DT
axs[0].plot(lm, crps["Extrapolation"], "s-", color="#d62728", ms=4, label="Extrapolation")
axs[0].plot(lm, crps["STEPS"], "o-", color="#1f77b4", ms=4, label="STEPS (20)")
mp.skill_axes(axs[0], ylabel="CRPS (mm h$^{-1}$)", title="(a) CRPS")
axs[0].legend(frameon=False, fontsize=7)
verification.plot_ROC(roc, axs[1], opt_prob_thr=True)
axs[1].set_title(f"(b) ROC, > {THR:g} mm h$^{{-1}}$", loc="left")
verification.plot_reldiag(rel, axs[2])
axs[2].set_title("(c) Reliability", loc="left")
verification.plot_rankhist(rnk, axs[3])
axs[3].set_title("(d) Rank histogram", loc="left")
mp.save_figure(fig, "nb4_fig04_ensemble_verification")
plt.show()

> 🔍 **การอ่านกราฟ**
> - **(a)** STEPS มี CRPS ต่ำกว่า extrapolation เพราะ ensemble "กระจายความเสี่ยง" เรื่องตำแหน่งของเซลล์
> - **(c)** ถ้าจุดอยู่ใต้เส้นทแยง แปลว่า ensemble ให้ความน่าจะเป็นสูงเกินจริง (over-forecasting) ส่วนกรณีที่ตัวอย่างน้อย (ความน่าจะเป็นสูง) จุดจะแกว่งมาก
> - **(d)** รูปตัว U แปลว่า spread แคบเกินไป
> - ⚠️ ทั้งหมดมาจาก **เหตุการณ์เดียว 3 ชั่วโมง** ผลจึงยังสรุปทั่วไปไม่ได้ งานวิจัยจริงต้องใช้หลายเหตุการณ์

---
## 🧪 แบบฝึกหัด

### 🧪 แบบฝึกหัด 4.1: จำนวน ensemble member กับความน่าจะเป็นที่เชียงของ

รัน STEPS ที่ issue time 11:45 UTC (`I0`) ด้วย **5, 10 และ 40 members** (ใช้ seed เดียวกัน) แล้วพล็อต **neighbourhood probability (5 km)** ที่เชียงของเทียบกับเวลา ทั้ง 4 ขนาด (รวม 20 members ที่มีอยู่แล้ว `p_nb[5]`)

**คำถาม:** ensemble ขนาดเล็กให้ความน่าจะเป็นที่ "หยาบ" อย่างไร? จำนวนเท่าใดจึงเพียงพอสำหรับคำเตือนระดับอำเภอ?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `e_n, _ = run_steps(I0, N_LEAD, n_ens=n)`
- `mx = tt.point_series(e_n, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max")` แล้ว `np.mean(mx > THR, axis=0)`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 4.1 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# fig, ax = plt.subplots(figsize=(5.5, 3))
# for n in [5, 10, 40]:
#     e_n, _ = run_steps(I0, N_LEAD, n_ens=n)
#     mx = tt.point_series(e_n, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max")
#     ax.plot(valid, np.mean(mx > THR, axis=0), label=f"{n} members")
# ax.plot(valid, p_nb[5], label="20 members"); ax.legend(); plt.show()

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 4.1 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
fig, ax = plt.subplots(figsize=(5.5, 3), layout="constrained")
for n, col in [(5, "#c6dbef"), (10, "#6baed6"), (20, "#2171b5"), (40, "#08306b")]:
    if n == 20:
        p = p_nb[5]
    else:
        e_n, _ = run_steps(I0, N_LEAD, n_ens=n)
        p = np.mean(tt.point_series(e_n, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max") > THR, axis=0)
    ax.plot(valid, p, "o-", ms=3, color=col, label=f"{n} members")
ax.set_ylim(0, 1.05)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
mp.skill_axes(ax, xlabel="Time (UTC)", ylabel="P (5-km neighbourhood)", title="Effect of ensemble size")
ax.legend(frameon=False, fontsize=7)
mp.save_figure(fig, "nb4_ex41_ensemble_size")
plt.show()

### 🧪 แบบฝึกหัด 4.2: เมื่อ nowcast พลาด: พายุลูกที่สองที่เชียงของ

ในกราฟ NB1 ข้อ 5 มีพายุลูกที่สองเข้าเชียงของช่วง **13:15–13:45 UTC** ให้รัน STEPS ที่ issue time **12:45 UTC** (index 7) แล้วทำกราฟ neighbourhood probability (5 km, > 5 mm/h) ที่เชียงของ และแผนที่ความน่าจะเป็นที่ +30 นาที พร้อม contour ค่าสังเกต

**คำถาม:** nowcast จับพายุลูกนี้ได้หรือไม่? ดูแผนที่ ณ 12:45 UTC แล้วอธิบายว่าพายุที่เข้าเชียงของ "มาจากไหน" (เป็นเซลล์เดิมที่เคลื่อนเข้ามา หรือเป็นเซลล์อ่อนที่ **ทวีความรุนแรงขึ้น (intensification)** หรือ **ก่อตัวใหม่ (convective initiation)**)? เหตุใด nowcasting แบบ extrapolation/STEPS จึงพยากรณ์กรณีเหล่านี้ได้ยาก?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `e7, _ = run_steps(7, 5)` และค่าสังเกต `R[8:13]`
- ใช้โค้ดในข้อ 4–5 เป็นแบบ เปลี่ยน `ens` → `e7`, `obs` → `R[8:13]`, `valid` → `ts[8:13]`
- ดูภาพ ณ issue time: `mp.radar_map(ax, R[7], meta, zoom=CK_BOX)`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 4.2 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# e7, _ = run_steps(7, 5)
# p7 = np.mean(tt.point_series(e7, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max") > THR, axis=0)
# fig, axs = mp.map_panels(1, 3, meta, panel_width=3.0, zoom=CK_BOX)
# ... (R[7] ณ issue time, P ที่ +30 นาที, R[9] ค่าสังเกต)

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 4.2 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
e7, _ = run_steps(7, 5)
p7 = np.mean(tt.point_series(e7, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max") > THR, axis=0)
print("P (5-km) ที่เชียงของ:", " ".join(f"{t:%H:%M}={p:.2f}" for t, p in zip(ts[8:13], p7)))
print("ค่าสังเกตสูงสุด (mm/h):", " ".join(f"{t:%H:%M}={v:.1f}" for t, v in zip(ts[8:13], obs_ck[8:13])))

fig, axs = mp.map_panels(1, 3, meta, panel_width=3.0, zoom=CK_BOX)
im = mp.radar_map(axs[0], R[7], meta, title=f"Observed at issue time {ts[7]:%H:%M}", zoom=CK_BOX)
P = excprob(e7[:, 1], THR, ignore_nan=True); P[~radar_mask] = np.nan
imp = mp.radar_map(axs[1], P, meta, ptype="prob", probthr=THR, title=f"P(R > {THR:g}), +30 min",
                   zoom=CK_BOX, gridline_labels=("bottom",), north=False)
mp.obs_contour(axs[1], R[9], meta, level=THR)
mp.radar_map(axs[2], R[9], meta, title=f"Observed {ts[9]:%H:%M}", zoom=CK_BOX,
             gridline_labels=("bottom",), north=False)
mp.add_colorbar(fig, im, axs, shrink=0.6)
fig.colorbar(imp, ax=list(axs), orientation="vertical", shrink=0.7, pad=0.01, label=f"P(R > {THR:g} mm h$^{{-1}}$)")
mp.panel_labels(axs)
mp.save_figure(fig, "nb4_ex42_missed_initiation")
plt.show()
# ณ 12:45 มีเซลล์อ่อน (< 5 mm/h) อยู่ทางตะวันตกของเชียงของ (ใกล้เชียงแสน) ซึ่งเคลื่อนมาทางตะวันออก
# และ "ทวีความรุนแรงขึ้น" จนเกิน 20 mm/h ที่เชียงของ ช่วง 13:15–13:30
# STEPS/S-PROG สมมติว่าเซลล์เล็กจะ "จางลง" ตาม AR model จึงไม่มีทางพยากรณ์การเติบโต/การก่อตัวใหม่ได้
# → ต้องใช้ข้อมูลเพิ่ม เช่น NWP, ดาวเทียม (cloud-top cooling) หรือ machine learning

### 🧪 แบบฝึกหัด 4.3: แผนที่ neighbourhood probability ทั้งพื้นที่

คำนวณ **neighbourhood probability** ของ $R > 10$ mm/h รัศมี 10 km **ทุก pixel** ที่ lead time +30 นาที แล้วทำแผนที่เทียบกับ pixel probability (2 panel, ซูม `CK_BOX`)

**คำถาม:** แผนที่ใดสื่อสารกับประชาชนหรือหน่วยงานท้องถิ่นได้ดีกว่า? ทำไม?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- ค่าสูงสุดในรัศมี 10 km ของแต่ละ member: ใช้ `scipy.ndimage.maximum_filter(field, footprint=disk)` โดย `disk` เป็นวงกลมรัศมี 10 pixel
- สร้าง disk: `yy, xx = np.mgrid[-10:11, -10:11]; disk = (xx**2 + yy**2) <= 100`
- จากนั้น `np.mean(mx > 10, axis=0)`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 4.3 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# from scipy.ndimage import maximum_filter
# yy, xx = np.mgrid[-10:11, -10:11]; disk = (xx**2 + yy**2) <= 100
# mx = np.stack([maximum_filter(np.nan_to_num(ens[m, 1]), footprint=disk) for m in range(ens.shape[0])])
# P_nb = ...
# P_px = excprob(ens[:, 1], 10.0, ignore_nan=True)
# fig, axs = mp.map_panels(1, 2, meta, panel_width=3.5, zoom=CK_BOX) ...

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 4.3 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
from scipy.ndimage import maximum_filter
yy, xx = np.mgrid[-10:11, -10:11]
disk = (xx ** 2 + yy ** 2) <= 100
k = 1
mx = np.stack([maximum_filter(np.nan_to_num(ens[m, k]), footprint=disk) for m in range(ens.shape[0])])
P_nb = np.mean(mx > 10.0, axis=0); P_nb[~radar_mask] = np.nan
P_px = excprob(ens[:, k], 10.0, ignore_nan=True); P_px[~radar_mask] = np.nan
fig, axs = mp.map_panels(1, 2, meta, panel_width=3.5, zoom=CK_BOX)
for j, (P, ttl) in enumerate([(P_px, "Pixel probability"), (P_nb, "10-km neighbourhood probability")]):
    im = mp.radar_map(axs[j], P, meta, ptype="prob", probthr=10, title=f"{ttl}, +30 min", zoom=CK_BOX,
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",), north=(j == 0))
    mp.obs_contour(axs[j], obs[k], meta, level=10.0)
mp.add_colorbar(fig, im, axs); mp.panel_labels(axs)
mp.save_figure(fig, "nb4_ex42_neighbourhood_map")
plt.show()

### 🧪 แบบฝึกหัด 4.4: (เสริม) LINDA-P

สร้าง **LINDA-P** ensemble 5 members ที่ issue time 11:45 UTC (`I0`) แล้วเปรียบเทียบ neighbourhood probability (5 km, > 5 mm/h) ที่เชียงของกับ STEPS
⚠️ ใช้เวลาประมาณ 2–3 นาที และ LINDA-P อาจให้ค่าฝนสูงสุดที่สูงผิดปกติในบาง member ให้ตรวจสอบค่าสูงสุดด้วย

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

```python
linda = nowcasts.get_method("linda")
lp = linda(np.nan_to_num(R[I0-2:I0+1]), V, N_LEAD, add_perturbations=True, n_ens_members=5,
           max_num_features=15, kmperpixel=1.0, timestep=15, seed=24, num_workers=2)
lp[..., ~radar_mask] = np.nan
```

</details>

In [ ]:
# ✍️ แบบฝึกหัด 4.4 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# linda = nowcasts.get_method("linda")
# lp = linda(...)
# p_lp = np.mean(tt.point_series(lp, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max") > THR, axis=0)

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 4.4 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
linda = nowcasts.get_method("linda")
with contextlib.redirect_stdout(io.StringIO()):
    lp = linda(np.nan_to_num(R[I0 - 2:I0 + 1]), V, N_LEAD, add_perturbations=True, n_ens_members=5,
               max_num_features=15, kmperpixel=1.0, timestep=15, seed=24, num_workers=2)
lp[..., ~radar_mask] = np.nan
print(f"ค่าฝนสูงสุด: LINDA-P = {np.nanmax(lp):.0f} mm/h | STEPS = {np.nanmax(ens):.0f} mm/h | obs = {np.nanmax(obs):.0f} mm/h")
p_lp = np.mean(tt.point_series(lp, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max") > THR, axis=0)
fig, ax = plt.subplots(figsize=(5.5, 3), layout="constrained")
ax.plot(valid, p_nb[5], "o-", color="#1f77b4", label="STEPS (20)")
ax.plot(valid, p_lp, "^-", color="#2ca02c", label="LINDA-P (5)")
ax.set_ylim(0, 1.05)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
mp.skill_axes(ax, xlabel="Time (UTC)", ylabel="P (5-km neighbourhood)", title="STEPS vs LINDA-P, Chiang Khong")
ax.legend(frameon=False, fontsize=7)
plt.show()

---
## 📝 สรุป
- STEPS ensemble ให้ **ความน่าจะเป็น** ซึ่งสื่อสารความไม่แน่นอนของพายุเซลล์เล็กได้ดีกว่า deterministic nowcast
- **Neighbourhood probability** เหมาะกับคำเตือนระดับพื้นที่ และลดปัญหาตำแหน่งผิดเล็กน้อย
- Nowcast **พยากรณ์การเติบโตหรือการก่อตัวใหม่ของพายุ** (intensification / convective initiation) ไม่ได้ ซึ่งเป็นข้อจำกัดสำคัญของพายุฤดูร้อนในประเทศไทย
- การประเมิน ensemble ต้องดูทั้ง CRPS, ROC, reliability และ rank histogram และต้องใช้หลายเหตุการณ์

### ❓ คำถามชวนคิด
1. ถ้าจะออก "คำเตือนลูกเห็บ" อัตโนมัติจาก nowcast ควรใช้ threshold กี่ dBZ และความน่าจะเป็นเท่าใด? จะหาค่าที่เหมาะสมอย่างไร (คำใบ้: ROC + ต้นทุนของการเตือนผิด/พลาด)
2. Ensemble ที่ spread แคบเกินไปจะส่งผลอย่างไรต่อความเชื่อมั่นของผู้ใช้?

➡️ **บทถัดไป:** Notebook 5 — ตรวจจับและติดตามเซลล์พายุ (T-DaTing): พายุลูกไหนผ่านเชียงของ?

### 📚 เอกสารอ้างอิง
- Bowler, N. E., Pierce, C. E., & Seed, A. W. (2006). STEPS. *Q. J. R. Meteorol. Soc.*, 132, 2127–2155.
- Schwartz, C. S., & Sobash, R. A. (2017). Generating probabilistic forecasts from convection-allowing ensembles using neighborhood approaches. *Mon. Wea. Rev.*, 145, 3397–3418.
- Hersbach, H. (2000). Decomposition of the CRPS for ensemble prediction systems. *Wea. Forecasting*, 15, 559–570.